# core

> An async Jupyter kernel client over HTTP and websockets

In [ ]:
#| default_exp core

jupyasyncclient talks to any server exposing the standard Jupyter kernels API: [rustygate](https://github.com/AnswerDotAI/rustygate), jupygate, or jupyter_server itself. Kernel lifecycle goes over HTTP (`POST /api/kernels`, interrupt, restart, delete); messaging goes over one websocket carrying the legacy Jupyter protocol (message dicts plus a `channel` key). There is no zmq anywhere in this package, and sends are genuinely awaited - the zmq subtleties (sync-send edge consumption, slow-joiner subscriptions, identity contracts) all live server-side.

The API mirrors jupyter_client's `AsyncKernelClient` where that helps a reader coming from it (`execute`, `kernel_info`, `wait_for_ready`), with three verbs at the core: `execute` sends fire-and-forget and returns the msg_id, `reply` awaits one `execute_reply`, and `run` collects every message one execute causes. Any `*_request` message type in the protocol is available as a generated method returning an awaitable of its reply. The kernel's asynchronous traffic (iopub, stdin, and the gateway-private cells channel) goes to one `on_jmsg` callback, in true arrival order, with each dict naming its origin in `channel`. The wire is already one ordered stream, and jupyter_client's per-channel queues only re-split what every consumer then re-merges. `JmsgQueues` is the pull adapter for consumers that read rather than accept calls; this page attaches one and reads the merged stream as `jmsg`.

Throughout this page the live server is a jupygate running an [ipymini](https://github.com/AnswerDotAI/ipymini) kernel; everything works identically against jupyter_server (the test suite runs against that).

In [ ]:
#| export
import asyncio, functools, inspect, json, logging, os, ssl, time, uuid, websockets
from fasttransport.core import AsyncTransport
from fastspec.oapi import OpenAPIClient, SpecParser
from fastspec.errors import APIError
from contextlib import suppress
from urllib.parse import urlencode, urlsplit, urlunsplit
from jupywire.session import Session, validate_string_dict, dumps, loads, serialize_binary_message, deserialize_binary_message
from jupywire.ops import EvalOps
from jupywire.route import RouterOps, DeadKernelError
from fastcore.basics import patch
from fastcore.xtras import dict2obj
from fastcore.net import urlread


In [ ]:
from fastcore.test import test_eq, test_fail, ExceptionExpected
from fastcore.nbio import msg2out
from jupywire.route import JmsgQueues, OUTPUT_MSGS, COMM_MSGS
from jupywire.ops import EvalError
from rustygate.tools import start_gateway


In [ ]:
#| export
log = logging.getLogger('jupyasyncclient')

## Wire codec

The legacy protocol's two encodings: JSON text frames, and a binary framing (count, offset table, JSON, then the raw buffers) for messages that carry `buffers`. The [jupygate docs](https://AnswerDotAI.github.io/jupygate/core.html) walk the byte layout; the codec (`dumps`/`loads`, `serialize_binary_message`/`deserialize_binary_message`) lives in `jupywire.session`, and here it just needs to round-trip, with inbound buffers as `memoryview`s, zero-copy off the frame.

In [ ]:
ses = Session(key=b'demo')
msg = ses.msg('comm_msg', dict(comm_id='c', data={}))
msg['channel'] = 'iopub'
msg['buffers'] = [b'raw']
back = deserialize_binary_message(serialize_binary_message(msg))
test_eq(bytes(back['buffers'][0]), b'raw')
test_eq(loads(dumps(dict(msg, buffers=[])))['content'], msg['content'])
back['channel']

'iopub'

## Shared HTTP plumbing

The three public classes all speak the same HTTP surface, so it lives in one base: URL joining (including the http-to-ws scheme flip for the channels endpoint), a [fasttransport](https://github.com/AnswerDotAI/fasttransport) `AsyncTransport` (a fresh client per request, or one you pass in), and token headers. The REST calls themselves are [fastspec](https://github.com/AnswerDotAI/fastspec) ops, generated from rustygate's spec, bundled with this package in fastspec's compact pre-parsed form (`rg_spec.py`) and sharing the base's transport, so `.api` on any client exposes the gateway's whole surface with signatures and docs straight from the spec. The kernels subset of that spec is the standard Jupyter kernels API, which is why the same ops work against jupyter_server too.

In [ ]:
#| export
def _join_url(base, path, ws=False, params=None):
    u = urlsplit(base)
    scheme = {"http": "ws", "https": "wss"}.get(u.scheme, u.scheme) if ws else u.scheme
    base_path = u.path.rstrip("/")
    full_path = f"{base_path}/{path.lstrip('/')}" if path else base_path
    query = urlencode({k: v for k, v in (params or {}).items() if v is not None})
    return urlunsplit((scheme, u.netloc, full_path, query, ""))

In [ ]:
#| export
@functools.cache
def rg_spec():
    "The bundled rustygate spec, loaded once per process from its compact `rg_spec` module."
    from jupyasyncclient.rg_spec import spec
    return SpecParser.from_dict(spec)

def build_spec(nm='jupyasyncclient/rg_spec.py', url='http://localhost:8787'):
    "Regenerate the compact spec module `rg_spec.py` from a running rustygate's `/openapi.json`."
    SpecParser.from_openapi(dict2obj(json.loads(urlread(f'{url}/openapi.json')))).save(nm)

class KernelApi:
    "Shared HTTP plumbing for the Jupyter kernels API: one transport, and the spec-generated ops on `api`."
    def __init__(self, base_url, token=None, headers=None, timeout=30, http_client=None, verify=True):
        self.base_url,self.token,self._timeout,self.verify = base_url.rstrip('/'),token or '',timeout,verify
        self._headers = {**(headers or {})}
        if self.token and 'Authorization' not in self._headers: self._headers['Authorization'] = f'token {self.token}'
        self.transport = AsyncTransport(timeout=timeout, client=http_client, base_headers=self._headers, verify=verify)
        self.api = OpenAPIClient(rg_spec(), transport=self.transport, base_url=self.base_url)

    def _ws_ssl(self, url):
        "An unverified ssl context when `verify=False` and `url` is wss, else None for the library default."
        if self.verify or not url.startswith('wss'): return None
        ctx = ssl.create_default_context()
        ctx.check_hostname,ctx.verify_mode = False,ssl.CERT_NONE
        return ctx

    def _kpath(self, kernel_id='', suffix=''): return f"/api/kernels/{kernel_id}{suffix}" if kernel_id else '/api/kernels'

## The client

`JupyAsyncKernelClient` is one kernel's worth of everything: the HTTP lifecycle calls for its kernel, the websocket connection, and the messaging API. The HTTP side comes first, since it can run before any websocket exists; then the private machinery that moves frames; then the public messaging surface, each method demonstrated as soon as it is defined.


In [ ]:
#| export
class JupyAsyncKernelClient(RouterOps, EvalOps, KernelApi):
    "AsyncKernelClient-ish API over the kernels HTTP API plus its websocket channels."
    allow_stdin = True
    def __init__(self, base_url, kernel_id=None, token=None, session_id=None, username=None, headers=None, timeout=30, http_client=None,
        reconnect=True, reconnect_ceiling=300.0, verify=True, max_size=256*2**20):
        super().__init__(base_url, token=token, headers=headers, timeout=timeout, http_client=http_client, verify=verify)
        self.kernel_id = kernel_id
        self.owned = False    # True only when `connect` created the kernel; honored by `__aexit__`
        self.session_id = session_id or uuid.uuid4().hex
        self.session = Session(session=self.session_id, username=username or os.environ.get("USER") or "")
        self._ws,self._start_task,self._send_task,self._recv_task,self._close_task = [None]*5
        self.reconnect,self.reconnect_ceiling,self._unsent,self._closing = reconnect,reconnect_ceiling,None,False
        self.max_size = max_size
        self._send_q = asyncio.Queue()
        self._init_router()

    def _kpath(self, kernel_id='', suffix=''): return super()._kpath(kernel_id or self.kernel_id, suffix)

`aclose` unwinds everything the client owns - loops, websocket, and pending futures; `stop_channels` is its fire-and-forget form for sync contexts.

In [ ]:
#| export
@patch
async def aclose(self: JupyAsyncKernelClient):
    self._closing = True
    if self._send_task and not self._send_task.done():   # the None sentinel ends `_send_loop` once the queue has drained
        self._send_q.put_nowait(None)
        with suppress(Exception):
            async with asyncio.timeout(2): await self._send_task
    for t in (self._start_task, self._send_task, self._recv_task):
        if t and not t.done(): t.cancel()
    if self._ws and self._ws.close_code is None: await self._ws.close()
    for t in (self._start_task, self._send_task, self._recv_task):
        if t:
            with suppress(asyncio.CancelledError, Exception): await t
    self.fail_waiters(RuntimeError('client closed'))
    self._ws = None

@patch
def stop_channels(self: JupyAsyncKernelClient):
    if self._close_task and not self._close_task.done(): return
    self._close_task = asyncio.create_task(self.aclose())

In [ ]:
#| export
@patch
async def start_kernel(self:JupyAsyncKernelClient, kernel_name="python3", **kwargs):
    model = await self.api.kernels.create_kernel(name=kernel_name, **kwargs)
    self.kernel_id = model["id"]
    return model

`start_kernel` posts to `/api/kernels` and binds the returned id. A live server to try it on - the rustygate binary, spawned in the background - and the rest of the page's client:


In [ ]:
g = start_gateway()
base_url = g.url
g


True

In [ ]:
kc = JupyAsyncKernelClient(base_url)
model = await kc.start_kernel()
model

{'id': '33873429da3d4fa38f77460c794d0f33',
 'name': '',
 'execution_state': 'alive',
 'connections': 0}

The remaining HTTP lifecycle calls are one-liners over the kernels ops. `model` fetches what the gateway reports about this kernel — its `execution_state`, `pid`, bound `path`, and so on ("kernel model" is jupyter_server's name for that dict). `is_alive` doubles as an existence check since the server 404s unknown kernel ids.

In [ ]:
#| export
@patch
async def shutdown_kernel(self: JupyAsyncKernelClient):
    "Delete the kernel, then close this client: with the kernel gone the connection has nothing left to serve"
    try:
        if self.kernel_id: return await self.api.kernels.delete_kernel(kid=self.kernel_id)
    finally: await self.aclose()
@patch
async def interrupt_kernel(self: JupyAsyncKernelClient):
    if self.kernel_id: return await self.api.kernels.interrupt(kid=self.kernel_id)
@patch
async def restart_kernel(self: JupyAsyncKernelClient):
    if self.kernel_id: return await self.api.kernels.restart(kid=self.kernel_id)


`model` and `is_alive` read what the gateway reports:

In [ ]:
#| export
@patch
async def model(self: JupyAsyncKernelClient):
    "What the gateway reports about this kernel (jupyter's `kernel model`): `execution_state`, `pid`, `path`, ..."
    return await self.api.kernels.get_kernel(kid=self.kernel_id)

@patch
async def is_alive(self: JupyAsyncKernelClient):
    if not self.kernel_id: return False
    try: return bool(await self.model())
    except Exception: return False

In [ ]:
m = await kc.model()
test_eq((m['id'], await kc.is_alive()), (kc.kernel_id, True))
none_kc = JupyAsyncKernelClient(base_url)
test_eq(await none_kc.is_alive(), False)
m

`api` exposes the gateway's whole surface as ops generated from the bundled rustygate OpenAPI spec: everything the wrapper methods above use, plus the endpoints that have no wrapper, such as search and exec. Displaying an op shows its signature and parameter docs, straight from the spec:

In [ ]:
test_eq((await kc.api.kernels.get_kernel(kid=kc.kernel_id))['id'], kc.kernel_id)
kc.api.kernels.restart

In [ ]:
#| export
@patch
async def kernel_for(self: KernelApi, path):
    "The non-dead kernel model bound to notebook `path`, or None"
    ms = await self.api.kernels.list_kernels(path=str(path))
    return next((m for m in ms if m['execution_state'] != 'dead'), None)

`kernel_for` answers "which kernel has this notebook?": the gateway's `path` filter on the kernel list, reduced to the one live holder. A dead kernel keeps its binding but loses the lookup, so reopening a notebook whose kernel died finds nothing and creates afresh. It patches `KernelApi` and addresses the list route directly, since the kernel client's own `kernel_request` fills in its `kernel_id`:

In [ ]:
b = JupyAsyncKernelClient(base_url)
await b.start_kernel(path='bound.ipynb')
found = await kc.kernel_for('bound.ipynb')
test_eq(found['id'], b.kernel_id)
test_eq(await kc.kernel_for('unbound.ipynb'), None)
await b.shutdown_kernel()

### How it works

Routing is `jupywire.route.RouterOps`, shared with `conkernelclient`; DESIGN.md in jupywire states the full contract. Every inbound frame goes to `route`. A message parented to a `run()` in flight is collected by that run. A shell or control message whose parent msg_id has a `reply()` or request future resolves it. Everything else goes to `on_jmsg`. A synthesized `status` message with `execution_state` `dead` fails every waiter first, so no caller sleeps through kernel death. The router also remembers the last `input_request` header, which `input()` uses below.


In [ ]:
#| export
@patch
async def _ensure_started(self: JupyAsyncKernelClient):
    if not self._start_task: self.start_channels()
    if self._start_task: await self._start_task

@patch
def send(self: JupyAsyncKernelClient, msg, channel: str):
    "Serialize `msg` (binary framing when it carries buffers) and queue its frame for `_send_loop`; returns the msg_id."
    msg = dict(msg)
    msg["channel"] = channel
    bufs = msg.get("buffers")
    if bufs:
        msg["buffers"] = [bytes(b) for b in bufs]
        payload = serialize_binary_message(msg)
    else:
        msg.pop("buffers", None)
        payload = dumps(msg)
    self._send_q.put_nowait(payload)
    return msg["header"]["msg_id"]

In [ ]:
#| export
@patch
async def _send_loop(self: JupyAsyncKernelClient):
    assert self._ws is not None
    while True:
        payload = self._unsent if self._unsent is not None else await self._send_q.get()
        if payload is None: return
        self._unsent = payload
        try: await self._ws.send(payload)
        except Exception as e:
            log.warning("websocket send failed: %s", e)
            return  # the payload stays in `_unsent`: a reconnected send loop retries it first
        self._unsent = None

@patch
async def _recv_loop(self: JupyAsyncKernelClient):
    assert self._ws is not None
    with suppress(websockets.ConnectionClosed):
        async for data in self._ws:
            if isinstance(data, str): msg = loads(data)
            elif isinstance(data, bytes): msg = deserialize_binary_message(data)
            else: continue
            r = self.route(msg)
            if inspect.isawaitable(r): await r
    if self.reconnect and not self._closing: self._start_task = asyncio.create_task(self._reconnect())

`_start_ws` dials the channels endpoint and starts a fresh pair of loops:

In [ ]:
#| export
@patch
async def _start_ws(self: JupyAsyncKernelClient):
    if self._ws and self._ws.close_code is None: return
    for t in (self._send_task, self._recv_task):
        if t and not t.done(): t.cancel()  # a stale send loop must not steal payloads from the new one
    params = {"session_id": self.session_id}
    if self.token: params["token"] = self.token
    ws_url = _join_url(self.base_url, self._kpath(suffix="/channels"), ws=True, params=params)
    self._ws = await websockets.connect(ws_url, ssl=self._ws_ssl(ws_url), additional_headers=self._headers, ping_interval=30, max_size=self.max_size)
    self._send_task = asyncio.create_task(self._send_loop())
    self._recv_task = asyncio.create_task(self._recv_loop())

Sending is the mirror image. `_exec_req` builds a signed message and queues its frame at call time, returning the msg_id, so wire order is call order. `send` is the seam under everything: it serializes one message dict and queues its frame. `__getattr__` makes every `*_request` message type in the protocol callable by name, returning an awaitable of its reply; the typed wrappers further down only add convenient defaults. `reply()` and `run()` come from `RouterOps`, built over `execute` and `send`.


In [ ]:
#| export
@patch
def _exec_req(self: JupyAsyncKernelClient, name, content=None, channel="shell", metadata=None, subshell_id=None, parent=None, msg_id=None, buffers=None):
    "Build a signed message and queue its frame, fire-and-forget; returns the msg_id."
    msg = self.session.msg(name, content, metadata=metadata, parent=parent)
    if buffers: msg["buffers"] = buffers
    if subshell_id: msg["header"]["subshell_id"] = subshell_id
    if msg_id: msg["header"]["msg_id"] = msg_id
    return self.send(msg, channel)

def _gen_request(self, name):
    "Generated `*_request` senders, each returning an awaitable of its reply; other names raise, so typos fail instead of sending bogus messages. Assigned onto the class below (a module-level `__getattr__` would become a PEP 562 hook)."
    if name.startswith("_") or not name.endswith("_request"): raise AttributeError(name)
    def _f(channel="shell", timeout=None, **kwargs): return self.request(name, kwargs or None, channel, timeout=timeout)
    return _f

JupyAsyncKernelClient.__getattr__ = _gen_request

### Connecting

In [ ]:
#| export
@patch
def start_channels(self: JupyAsyncKernelClient, shell=True, iopub=True, stdin=True, control=True):
    if not (shell or iopub or stdin or control): return
    if self._start_task and not self._start_task.done(): return
    self._start_task = asyncio.create_task(self._start_ws())
    return self

In [ ]:
#| export
@patch(as_prop=True)
def channels_running(self: JupyAsyncKernelClient): return bool(self._ws and self._ws.close_code is None)

`start_channels` opens the websocket (session id and token as query params) and starts the loops; `channels_running` reports the live state. It returns immediately: readiness is `wait_for_ready`'s job.


`route` hands everything unmatched to `on_jmsg`, and a client with no callback drops it. Pull-style consumers attach `JmsgQueues`, which registers itself as the callback and queues each message by channel, raising `queue.Empty` on a `get` timeout. This page attaches one below, folding iopub, stdin, and the gateway's cells channel into one `jmsg` queue. `cells` carries a gateway's notebook change broadcasts (`cell_ops` messages); the files page demonstrates it.


In [ ]:
#| export
@patch
async def wait_for_ready(self: JupyAsyncKernelClient, timeout=None):
    await self._ensure_started()
    await self.kernel_info_request(timeout=timeout)

`wait_for_ready` is one `kernel_info` round trip. On a reattach the gateway replays the frames this `session_id` missed, and readiness must not eat them; the `kernel_info` status chatter goes to `on_jmsg`, harmless because a handler tolerates message types it does not use. Over a websocket readiness is purely about the kernel being up: there is no subscription race to compensate for, since the gateway handled zmq subscription on its side. It calls the generated `kernel_info_request` rather than the typed wrapper defined further down, so the machinery above is all it needs.


In [ ]:
kc.start_channels()
await kc.wait_for_ready(timeout=60)
kc.channels_running


True

A raw generated sender, awaited (`comm_info_request` has no typed wrapper here):


In [ ]:
m = await kc.comm_info_request(timeout=15)
m['msg_type']


### The request API

In [ ]:
#| export
@patch
def execute(self: JupyAsyncKernelClient, code, silent=False, store_history=True, user_expressions=None, allow_stdin=None, stop_on_error=True,
    msg_id=None, metadata=None, subshell_id=None, buffers=None):
    "Send an `execute_request`, fire-and-forget; returns its msg_id."
    user_expressions = {} if user_expressions is None else user_expressions
    allow_stdin = self.allow_stdin if allow_stdin is None else allow_stdin
    if not isinstance(code, str): raise ValueError(f"code {code!r} must be a string")
    validate_string_dict(user_expressions)
    return self._exec_req("execute_request", dict(code=code, silent=silent, store_history=store_history, user_expressions=user_expressions,
        allow_stdin=allow_stdin, stop_on_error=stop_on_error), metadata=metadata, subshell_id=subshell_id, msg_id=msg_id, buffers=buffers)

The lessons below read the merged stream, so attach the pull adapter now. An execute nobody awaits broadcasts its output into it:


In [ ]:
qs = JmsgQueues(kc, queues=('shell', 'control', 'jmsg'), merge=dict(iopub='jmsg', stdin='jmsg', cells='jmsg'))
kc.execute("print('unread')")
s = await qs.jmsg_for('stream', timeout=15)
s['content']['text']


`reply` awaits the `execute_reply`; the execute's iopub arrives independently through `on_jmsg`.


In [ ]:
rep = await kc.reply("print('hello'); 6*7", timeout=30)
test_eq(rep['content']['status'], 'ok')
s = await qs.jmsg_for('stream', timeout=15)
r = await qs.jmsg_for('execute_result', timeout=15)
dict(stream=s['content']['text'].strip(), result=r['content']['data']['text/plain'])


`pred=` narrows beyond the type. The execute above has left its `status` chatter queued; the predicate picks out the idle:

In [ ]:
idle = await qs.jmsg_for('status', pred=lambda m: m['content']['execution_state']=='idle', timeout=15)
idle['content']['execution_state']


Requests are concurrency-safe: fire several, gather the replies; each future resolves from its own parent msg_id. Every request is on the wire before its call returns, so submission order never depends on await order.

In [ ]:
reps = await asyncio.gather(*[kc.reply(f'{i}*{i}', timeout=30) for i in range(5)])
[r['content']['status'] for r in reps]


A host can choose the request's msg_id rather than accepting a generated one: `msg_id=` on `execute`, `reply`, or `run` becomes the header id, and every message the request produces carries it back in `parent_header`. That is how a host attributes streamed output to its own units of work, e.g. solveit's `{message_id}.{token}` scheme:

In [ ]:
rep = await kc.reply("21*2", timeout=30, msg_id='myid.abc123')
test_eq(rep['parent_header']['msg_id'], 'myid.abc123')


A `reply` await that gives up cleans up: timing out (or being cancelled) pops the pending entry, so a late `execute_reply` is claimed by nobody and reaches `on_jmsg` with the rest of the unrouted traffic:

In [ ]:
lid = f'late.{uuid.uuid4().hex[:6]}'
w = kc.reply('import time; time.sleep(1)', timeout=0.25, msg_id=lid)
with ExceptionExpected(TimeoutError): await w
assert lid not in kc.replies
late = await qs.jmsg_for('execute_reply', queue='shell', pred=lambda m: m['parent_header']['msg_id']==lid, timeout=15)
late['msg_type']

Subshells (JEP 91) pass through gateway and kernel untouched: create one on the control channel, and the main shell keeps answering while the subshell is busy; `list_subshell` names the live ids.

In [ ]:
#| export
@patch
def create_subshell(self: JupyAsyncKernelClient, timeout=None): return self.create_subshell_request(channel="control", timeout=timeout)

@patch
def list_subshell(self: JupyAsyncKernelClient, timeout=None): return self.list_subshell_request(channel="control", timeout=timeout)

@patch
def delete_subshell(self: JupyAsyncKernelClient, subshell_id: str, timeout=None):
    return self.delete_subshell_request(subshell_id=subshell_id, channel="control", timeout=timeout)

In [ ]:
sub = (await kc.create_subshell(timeout=15))['content']['subshell_id']
assert sub in (await kc.list_subshell(timeout=15))['content']['subshell_id']
busy = kc.reply('import time; time.sleep(1.5)', timeout=30, subshell_id=sub)
matches,_ = await kc.complete('imp', timeout=1)
rep = await busy
await kc.delete_subshell(sub, timeout=15)
rep['content']['status'], rep['parent_header']['subshell_id']==sub, 'import' in matches


In [ ]:
#| export
@patch
def release(self: JupyAsyncKernelClient, msg_id: str, status: str = "ok", timeout=None):
    "Complete a held execute (kernmini's `hold` metadata); `status='error'` makes the hold's reply an error, engaging the kernel's stop-on-error tail abort"
    return self.release_request(msg_id=msg_id, status=status, channel="control", timeout=timeout)

A held execute — kernmini's `hold` execute-metadata — parks the kernel's queue while work happens outside the kernel, and `priority` metadata lets designated requests overtake whatever is queued (highest first, FIFO within a level). Both ride ordinary execute metadata, so they pass through the gateway untouched; `release` completes the hold from the control channel, and `found` reports whether it was still parked (a late release after a timeout is a quiet no-op):

In [ ]:
hid = f'hold1.{uuid.uuid4().hex[:6]}'
kc.execute('', metadata=dict(hold=True), msg_id=hid)
tail = kc.reply("'after the hold'", timeout=30)
jump = await kc.reply("'jumped'", timeout=30, metadata=dict(priority=1))
test_eq(jump['content']['status'], 'ok')
rel = await kc.release(hid, timeout=15)
test_eq(rel['content']['found'], True)
test_eq((await tail)['content']['status'], 'ok')


A hold released with `status='error'` aborts the queued tail, exactly as a cell error does under `stop_on_error` — and the aborted requests still resolve every waiting `reply()`, with `status='aborted'`. Shell and control diverge at the gateway's zmq hop, so a short pause makes sure the tail is queued before the release lands:

In [ ]:
hid = f'hold2.{uuid.uuid4().hex[:6]}'
kc.execute('', metadata=dict(hold=True), msg_id=hid)
t2 = kc.reply("'never runs'", timeout=30)
await asyncio.sleep(0.2)
await kc.release(hid, status='error', timeout=15)
test_eq((await t2)['content']['status'], 'aborted')


`input()` answers the pending prompt, parented to it (the router remembered the `input_request` header).

In [ ]:
fut = asyncio.ensure_future(kc.reply("name = input('who? ')", timeout=30))
prompt = await qs.jmsg_for('input_request', timeout=15)
kc.input('Jeremy')
await fut
rep = await kc.reply('name', timeout=30, user_expressions={'v':'name'})
rep['content']['user_expressions']['v']['data']['text/plain']


The raw `*_request` senders mirror the wire protocol, each returning an awaitable of its reply. `request` is the same thing with a name argument instead of a generated attribute, and `shell` and `control` name its channel. It is the seam `RouterOps`'s typed verbs are written over, so `complete`, `inspect`, `check`, `history`, and the `comm_msg` sender all come from `jupywire.route` - the ergonomic values a frontend actually wants (unwrapped results, cursor position defaulted to the end), identical across this client and `conkernelclient`:


In [ ]:
matches, start = await kc.complete('imp')
assert 'import' in matches
matches[:3]


['import']

In [ ]:
txt = await kc.inspect('print')
assert 'print' in txt
test_eq(await kc.inspect('no_such_name_here'), '')


True

In [ ]:
h = await kc.history(timeout=15)
len(h['content']['history']) > 0

True

`comm_info` and `kernel_info` keep the conventional wrapper shape; `check` (over `is_complete_request`) is the one frontends poll while the user types:


In [ ]:
#| export
@patch
def comm_info(self: JupyAsyncKernelClient, target_name=None, timeout=None):
    return self.comm_info_request(target_name=target_name, timeout=timeout)

@patch
def kernel_info(self: JupyAsyncKernelClient, timeout=None): return self.kernel_info_request(timeout=timeout)

In [ ]:
test_eq(await kc.check('for i in range(3):'), ('incomplete', '    '))
test_eq((await kc.check('1+1'))[0], 'complete')


'incomplete'

### Calling kernel functions

`reply` comes from `RouterOps`: it sends the execute at call time and returns an awaitable of the `execute_reply`. `eval` turns the `user_expressions` round trip into a function call: run `func(*args, **kw)` kernel-side (awaiting coroutines), bring the result back by repr, and reconstruct it client-side — `try_eval` wraps primitive results in a dynamic class named after the kernel-side type. `_call=False` evaluates `func` as a bare expression. `ipy` targets `get_ipython()` methods, and the generated service methods mirror what `ipyfuncs` patches onto the kernel's shell (`sig_help`, `get_schemas`, `ranked_complete`, ...). The whole family is inherited from jupywire's `EvalOps` mixin over this module's `reply` — one definition shared with `conkernelclient`, so callers read identically over zmq and websockets. `priority=` sends kernmini's `priority: 1` execute metadata, so the call overtakes queued work (the held turn below shows it); the service methods default it on.

In [ ]:
r = await kc.reply('def add(a, b): return a+b')
test_eq(r['content']['status'], 'ok')
test_eq(await kc.eval('add', a=10, b=20), 30)
await kc.reply('a = [1,2,3]')
test_eq(await kc.eval('a', _call=False), [1,2,3])
test_eq(await kc.eval('add', a=30, b=40, _literal=False), '70')
_r = await kc.eval('missing_fn')
assert 'NameError' in _r

`eval_expr` (also shared, from `EvalOps`) is the expression-only sibling: no function call machinery, just one `user_expressions` round trip, the repr parsed back via `literal_eval` where its form allows. Unlike `eval`, a kernel-side error raises `EvalError` rather than returning the traceback as a string:

In [ ]:
test_eq(await kc.eval_expr('a[-1] * 2'), 6)
with ExceptionExpected(EvalError, 'NameError'): await kc.eval_expr('no_such_name')


The `_ipy_funcs` services live kernel-side in [`ipyfuncs`](https://github.com/AnswerDotAI/ipyfuncs); importing it is the whole setup, so the battery is self-sufficient:

In [ ]:
await kc.reply('import ipyfuncs')
await kc.reply('''def range_ex(
    a:str  # some param
):
    "some func docstring"
    ...''')
sigs = await kc.sig_help(code='range_ex(', line_no=1, col_no=9)
test_eq(sigs[0]['label'], 'range_ex')
schemas = await kc.get_schemas(fs=['range_ex'])
test_eq(schemas['range_ex']['function']['name'], 'range_ex')


`xpush`, `retr`, `eval_exprs`, and `xenv` round-trip values and environment variables:

In [ ]:
kc.xpush(asdf=4)
test_eq(await kc.retr('asdf'), 4)
test_eq(await kc.eval_exprs(vs=['list(range(5))']), {'list(range(5))': [0,1,2,3,4]})
kc.xenv(hi='jupy')
test_eq(await kc.eval('__os.environ["hi"]', _call=False), 'jupy')

### Collecting an execution: `run`

`execute` returns when the request is sent, and `reply` returns one message; iopub arrives independently, any time, interleaved with every other client's traffic. A REPL-shaped consumer wants "*this* execution's messages, in order, until it finishes". `run` (from `RouterOps`, shared with `conkernelclient`) owns that view. It sends when awaited, files its entry before sending, and collects every message parented to the execute - outputs, statuses, the `execute_reply` - completing when both the `execute_reply` and the idle status have arrived. It returns the raw messages, and `exec_outs` is the rendered form: just the nbformat outputs, via `msg2out`. Concurrent runs each collect only their own traffic, and a dead kernel raises `DeadKernelError` in every waiting run.


The collected view, raw and rendered:


In [ ]:
msgs = await kc.run("print('hi'); 6*7")
test_eq([m['msg_type'] for m in msgs if m['msg_type'] in OUTPUT_MSGS], ['stream', 'execute_result'])
outs = await kc.exec_outs("print('hi'); 6*7")
test_eq([o['output_type'] for o in outs], ['stream', 'execute_result'])
outs


`on_output` streams each raw message to a callback as it arrives, for consumers that render live (clikernel's stream worker is one):

In [ ]:
types = []
await kc.run("print('x')", on_output=lambda m: types.append(m['msg_type']))
types

An error is an ordinary `error` output plus the reply's verdict, so a consumer chooses its own severity:


In [ ]:
msgs = await kc.run('1/0')
rep = next(m for m in msgs if m['msg_type'] == 'execute_reply')
test_eq(rep['content']['status'], 'error')
outs = [msg2out(m) for m in msgs if m['msg_type'] in OUTPUT_MSGS]
test_eq(outs[0]['ename'], 'ZeroDivisionError')


An `input_request` from the running cell bypasses the run and reaches `on_jmsg`, so one handler answers prompts for `execute` and `run` alike:


In [ ]:
task = asyncio.ensure_future(kc.run("nm = input('who? ')"))
prompt = await qs.jmsg_for('input_request', timeout=15)
kc.input('jupy')
await task
test_eq(await kc.retr('nm'), 'jupy')


Comm traffic a cell produces is parented to its execute like any other message, so a `run` collects it; `COMM_MSGS` names the three types, and none of them is an output. Comms are how a kernel-side magic talks to its host app (e.g. ipyai's `%ipyai`). Comm traffic parented to nothing this client is running goes to `on_jmsg` for the host:


In [ ]:
msgs = await kc.run("from comm import create_comm\ncm = create_comm('demo', data=dict(x=1))\ncm.send(dict(y=2))")
comms = [(m['msg_type'], m['content'].get('data')) for m in msgs if m['msg_type'] in COMM_MSGS]
test_eq(comms[0], ('comm_open', dict(x=1)))
test_eq(comms[1], ('comm_msg', dict(y=2)))
test_eq([m for m in msgs if m['msg_type'] in OUTPUT_MSGS], [])


The filtering contract, live: iopub is a shared broadcast, so an out-of-band execute (a host's silent bridge exec, another client) broadcasts its own outputs and its own `idle` while a run is collecting. Neither may leak in: a foreign output must not appear in the run, and a foreign idle must not end it early. The foreign traffic goes to `on_jmsg` instead:


In [ ]:
kc.execute("'FOREIGN'")
outs = await kc.exec_outs("'MINE'")
test_eq(len(outs), 1)
assert 'MINE' in outs[0]['data']['text/plain'] and 'FOREIGN' not in str(outs)
foreign = await qs.jmsg_for('execute_result', timeout=15)
assert 'FOREIGN' in str(foreign['content'])


Runs are concurrency-safe: each files its entry before sending, so several in flight on one client each collect only their own traffic, however the kernel's queue interleaves it (gathered with a bound, so a routing regression fails instead of hanging):


In [ ]:
outs = await asyncio.wait_for(asyncio.gather(kc.exec_outs('1+1'), kc.exec_outs("print('two')"), kc.exec_outs('3+3')), 30)
test_eq([o[0]['data']['text/plain'] for o in (outs[0], outs[2])], ['2', '6'])
test_eq(outs[1][0]['text'], 'two\n')


Concurrent submission means the kernel queues the cells, and with the wire default `stop_on_error=True` an error in one aborts the runs queued behind it (status `aborted`). Independent runs pass `stop_on_error=False`:

In [ ]:
o1, o2, o3 = await asyncio.wait_for(asyncio.gather(*[kc.exec_outs(c, stop_on_error=False) for c in ('y = 6*7', '1/0', 'y')]), 30)
test_eq((o1, o2[0]['ename'], o3[0]['data']['text/plain']), ([], 'ZeroDivisionError', '42'))


## Protocol details

Lessons that apply to any client of these kernels: payload replies, display metadata and transients, display-id updates, and what an interrupt produces.

Payloads are used to add dicts directly into the message reply. The `source` key says what kind of payload it is. There are some officially recognised sources, like `page` to show stuff in a pager, of `set_next_input` to add a new cell under the current one.

For instance, this is extended by the aimagic nbextension to add `ctype` and `offset` keys:

```py
pm = get_ipython().payload_manager
pm.write_payload(dict(
    source='set_next_input', text='bar',
    ctype='markdown', replace=False, offset=1))
```

We can invent any source we like, and that will be included in the message. With `single=False` we can accumulate multiple messages:

In [ ]:
code = '''payl = dict(source='testing', foo='bar')
pm = get_ipython().payload_manager
pm.write_payload(payl, single=False)
pm.write_payload(dict(source='testing', foo='baz'), single=False)'''
r = await kc.reply(code, timeout=30)
test_eq([p['foo'] for p in r['content']['payload']], ['bar','baz'])

We can add metadata and "transient"s (data that should not be persisted in a notebook) to display objects by adding params to `display`:

In [ ]:
kc.execute("from IPython.display import display\ndisplay('hi', metadata={'key':'value'}, transient={'foo':'bar'})")
dd = await qs.jmsg_for('display_data', timeout=15)
test_eq(dd['content']['metadata'], {'key':'value'})
test_eq(dd['content']['transient'], {'foo':'bar'})

A display with a `display_id` can be updated later: the follow-up arrives as `update_display_data` carrying the same id in its `transient`, and a client that tracks ids replaces the earlier output instead of appending:

In [ ]:
code = '''from IPython.display import display
display('first', display_id='qqww')
display('second', update=True, display_id='qqww')'''
kc.execute(code)
upd = await qs.jmsg_for('update_display_data', timeout=15)
test_eq(upd['content']['transient'], {'display_id':'qqww'})
test_eq(upd['content']['data']['text/plain'], "'second'")

An interrupt (`interrupt_request` on the control channel, or the HTTP `interrupt` endpoint) reaches the running cell as an ordinary `error` output, `KeyboardInterrupt`:

In [ ]:
task = asyncio.ensure_future(kc.exec_outs("import time; time.sleep(10); print('finished')"))
await asyncio.sleep(0.3)
await kc.interrupt_kernel()
outs = await task
test_eq(outs[0]['ename'], 'KeyboardInterrupt')


### Reconnecting

Over zmq a dropped connection is invisible: the transport redials by itself and jupyter_client never knows. A websocket client has to earn that property. When the receive loop ends without `aclose` having been called, the client redials `/channels` with the same `session_id`, which is what lets a gateway hand back the surviving queue and replay what was missed (see the gateway's reconnect contract); the send loop resumes behind it, and a frame that died mid-send is resent first. Pending `reply()` futures deliberately survive the drop: the gateway replays the very replies they await, so failing them would be premature.

Giving up is a separate decision from retrying. Each failed redial probes the kernel over HTTP: if the server answers and the kernel is gone, retrying would be a lie, so pending futures fail immediately with `DeadKernelError`. If the server itself is unreachable, the client retries with backoff until `reconnect_ceiling` seconds have passed, then fails them with `ConnectionError`. `reconnect=False` restores fail-fast behavior.

In [ ]:
#| export
@patch
async def _reconnect(self: JupyAsyncKernelClient):
    "Redial with the same session id, with backoff; on a dead kernel or an expired ceiling, fail every pending reply and in-flight run."
    deadline, delay = time.monotonic() + self.reconnect_ceiling, 0.1
    while not self._closing:
        try:
            await self._start_ws()
            log.info("websocket reconnected")
            return
        except Exception as e:
            exc = None
            try: await self.model()
            except APIError as he:
                if he.status_code: exc = DeadKernelError(f'kernel {self.kernel_id} is gone: {he.message}')
            except Exception: pass  # the server is unreachable too: keep trying until the ceiling
            if exc is None and time.monotonic() > deadline: exc = ConnectionError(f'gave up reconnecting after {self.reconnect_ceiling}s: {e}')
            if exc:
                self.fail_waiters(exc)
                raise exc
            await asyncio.sleep(delay)
            delay = min(delay*2, 5.0)

In [ ]:
fut = asyncio.ensure_future(kc.reply("import time; time.sleep(1); 'survived'", timeout=20))
await asyncio.sleep(0.3)
kc._ws.transport.abort()
rep = await fut
test_eq(rep['content']['status'], 'ok')
test_eq((await kc.reply('1+1', timeout=15))['content']['status'], 'ok')
kc.channels_running


A `run()` in flight survives the drop the same way: its entry keeps collecting, and the gateway replays the messages the dead socket lost:

In [ ]:
task = asyncio.ensure_future(kc.run("import time; time.sleep(1); 'collected'"))
await asyncio.sleep(0.3)
kc._ws.transport.abort()
msgs = await asyncio.wait_for(task, 30)
test_eq(next(m['content']['status'] for m in msgs if m['msg_type'] == 'execute_reply'), 'ok')

Kernel death arrives as rustygate's synthesized `status` message with `content.execution_state` `dead`, the only death broadcast a websocket client gets. It fails every pending `reply()` and `run()` with `DeadKernelError` and then reaches `on_jmsg`. The reconnect probe reaches the same verdict when the websocket drops first. A kernel whose process dies mid-cell never replies:

In [ ]:
d = JupyAsyncKernelClient(base_url)
await d.start_kernel()
d.start_channels()
await d.wait_for_ready(timeout=60)
w = d.reply('import os, signal; os.kill(os.getpid(), signal.SIGKILL)', timeout=30)
with ExceptionExpected(DeadKernelError): await w
await d.aclose()


Messages have a size ceiling, `max_size` (default 256MB, `None` to disable), passed to the websocket library; the library closes the connection when a message exceeds it.


### Closing down

`shutdown` goes over the control channel; the HTTP `DELETE` (via `shutdown_kernel`) is the more common route since it also reaps the server-side process.

In [ ]:
#| export
@patch
def shutdown(self: JupyAsyncKernelClient, restart=False, timeout=None):
    return self.shutdown_request(restart=restart, channel="control", timeout=timeout)

`aclose` flushes queued outbound frames before tearing down, bounded at 2s. A fire-and-forget request followed by an immediate close still sends. `shutdown_kernel` deletes the kernel and then closes this client itself, since with the kernel gone the connection has nothing left to serve. The common full teardown is the one call:

The parked queue survives a full disconnect too. A client that goes away entirely and later redials with the same `session_id` receives everything the kernel produced in between. Readiness must not eat the replay, and the reader attaches its `JmsgQueues` before connecting, so no replayed frame is dropped:

In [ ]:
a = JupyAsyncKernelClient(base_url, session_id='parked-demo')
await a.start_kernel()
a.start_channels()
await a.wait_for_ready(timeout=60)
a.execute("import time; time.sleep(0.5); print('parked')")
await a.aclose()
await asyncio.sleep(1)


Reconnect with the same `session_id` and read what was missed:

In [ ]:
b = JupyAsyncKernelClient(base_url, session_id='parked-demo')
b.kernel_id = a.kernel_id
bq = JmsgQueues(b, queues=('jmsg',), merge=dict(iopub='jmsg'))
b.start_channels()
await b.wait_for_ready(timeout=60)
m = await bq.jmsg_for('stream', timeout=15)
test_eq(m['content']['text'], 'parked\n')
await b.shutdown_kernel()

### The packaged startup

Construct, create, open channels, wait ready is the four-line dance every consumer types; `connect` packages it, covering attach with the same verb (`kernel=` an existing id). Ownership is recorded where it's decided: a kernel `connect` *created* is `owned`, and the context manager honors that on the way out — the client always closes, but the kernel is shut down only when owned. An attached kernel is never stopped, and bare `aclose` never kills anything: explicit `shutdown_kernel` remains the only other way a kernel ends.

In [ ]:
#| export
@patch(cls_method=True)
async def connect(cls:JupyAsyncKernelClient, base_url, kernel=None, token=None, timeout=60, verify=True, **kw):
    "Construct + create a kernel (or attach to `kernel`) + open channels + wait ready; a created kernel is `owned`"
    self = cls(base_url, kernel_id=kernel, token=token, verify=verify)
    if kernel is None:
        await self.start_kernel(**kw)
        self.owned = True
    self.start_channels()
    await self.wait_for_ready(timeout=timeout)
    return self

@patch
async def __aenter__(self:JupyAsyncKernelClient): return self

@patch
async def __aexit__(self:JupyAsyncKernelClient, *exc):
    if self.owned:
        with suppress(Exception): await self.shutdown_kernel()
    else: await self.aclose()

In [ ]:
async with await JupyAsyncKernelClient.connect(base_url) as k2:
    kid2 = k2.kernel_id
    assert k2.owned and k2.channels_running
assert not await JupyAsyncKernelClient(base_url, kernel_id=kid2).is_alive()

Attaching with the same verb: `kernel=` binds an existing id, `owned` stays False, and leaving the block leaves the kernel running:

In [ ]:
async with await JupyAsyncKernelClient.connect(base_url, kernel=kc.kernel_id) as att: assert not att.owned
test_eq(await kc.is_alive(), True)

### TLS

A gateway started with `--tls` serves https and wss from a self-signed certificate it generates at boot, and `verify=False` accepts it: httpx skips certificate checks and the websocket uses an unverified context. This trades server identity for zero provisioning, the right deal for a box you own; token auth works unchanged either way.

In [ ]:
gs = start_gateway(tls=True)
assert gs.url.startswith('https://')
async with await JupyAsyncKernelClient.connect(gs.url, verify=False) as ks: rep = await ks.reply('6*7', timeout=30)
test_eq(rep['content']['status'], 'ok')
gs.stop()
gs

In [ ]:
await kc.shutdown_kernel()
test_eq(kc.channels_running, False)
test_eq(await none_kc.is_alive(), False)
await none_kc.aclose()


In [ ]:
#| hide
g.stop()


In [ ]:
#| hide
import nbdev
nbdev.nbdev_export()